# Experiments

### Project Setup

In [1]:
!git clone https://github.com/stanfordnlp/stanza-train.git

fatal: destination path 'stanza-train' already exists and is not an empty directory.


In [2]:
!pip install stanza==1.11.0

In [3]:
#!source /content/stanza-train/config/config.sh
!source stanza-train/config/config.sh

In [4]:
import os
#os.environ["UDBASE"] = "/content/stanza-train/data/udbase"
os.environ["UDBASE"] = "stanza-train/data/udbase"

In [5]:
!pip install --upgrade torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118


In [6]:
!pip install --upgrade transformers

In [7]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU name: None


In [8]:
# remove the UD_English-TEST folder from the training data
import shutil

try:
    #shutil.rmtree("/content/stanza-train/data/udbase/UD_English-TEST")
    shutil.rmtree("stanza-train/data/udbase/UD_English-TEST")
    print(f"Directory and all its contents removed.")
except Exception as e:
    print(f"Error: {e}")

Error: [Errno 2] No such file or directory: 'stanza-train/data/udbase/UD_English-TEST'


In [9]:
import time

### Model Training

In [10]:
#UD_PATHS = [entry.name for entry in os.scandir("/content/stanza-train/data/udbase") if entry.is_dir() and entry.name.startswith("UD_")]
UD_PATHS = [entry.name for entry in os.scandir("stanza-train/data/udbase") if entry.is_dir() and entry.name.startswith("UD_")]

In [11]:
UD_PATHS

['UD_Italian-ISDT', 'UD_German-GSD', 'UD_Afrikaans-AfriBooms', 'UD_Greek-GUD']

In [12]:
# create folder to hold all the logs from the commands we will be running
#os.makedirs("/content/logs", exist_ok=True)
os.makedirs("logs", exist_ok=True)

run fasttext embeds for each language

In [ ]:
# concatenate the txt files to a new file
# if the combined_data file exists it fails to complete
for UD_PATH in UD_PATHS:
    !cat stanza-train/data/udbase/{UD_PATH}/*.txt > stanza-train/data/udbase/{UD_PATH}/combined_data.txt

In [15]:
!git clone https://github.com/facebookresearch/fastText.git


fatal: destination path 'fastText' already exists and is not an empty directory.


In [16]:
%cd fastText/

/Users/maria/Documents/DPMS/dialectometry/dialectometry-distances/fastText


In [17]:
!make

make: Nothing to be done for `opt'.


In [18]:
os.makedirs("embeds", exist_ok=True)

In [19]:
for UD_PATH in UD_PATHS:
    !./fasttext skipgram -input /Users/maria/Documents/DPMS/dialectometry/dialectometry-distances/stanza-train/data/udbase/{UD_PATH}/combined_data.txt -output embeds/{UD_PATH}

Read 0M words
Number of words:  139
Number of labels: 0
Progress: 100.0% words/sec/thread:   26807 lr:  0.000000 avg.loss:  4.114141 ETA:   0h 0m 0s
Read 0M words
Number of words:  106
Number of labels: 0
Progress: 100.0% words/sec/thread:   24024 lr:  0.000000 avg.loss:  3.953825 ETA:   0h 0m 0s
Read 0M words
Number of words:  181
Number of labels: 0
Progress: 100.0% words/sec/thread:   32870 lr:  0.000000 avg.loss:  3.428340 ETA:   0h 0m 0s
Read 0M words
Number of words:  104
Number of labels: 0
Progress: 100.0% words/sec/thread:   22025 lr:  0.000000 avg.loss:  4.119097 ETA:   0h 0m 0s


run tokenizer for each language

In [20]:
%cd ..

/Users/maria/Documents/DPMS/dialectometry/dialectometry-distances


In [21]:
!pwd

/Users/maria/Documents/DPMS/dialectometry/dialectometry-distances


In [22]:
for UD_PATH in UD_PATHS:
  print(f"Running tokenization for {UD_PATH}")
  start_time = time.perf_counter()
  # changed the logs file
  !python3 -m stanza.utils.datasets.prepare_tokenizer_treebank {UD_PATH} &> logs/{UD_PATH}_prepare_tokenizer_log.txt
  !python3 -m stanza.utils.training.run_tokenizer {UD_PATH} --steps 1000 &> logs/{UD_PATH}_run_tokenizer_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running tokenization for UD_Italian-ISDT
Total time taken: 3.5940 seconds
Running tokenization for UD_German-GSD
Total time taken: 1498.0779 seconds
Running tokenization for UD_Afrikaans-AfriBooms
Total time taken: 2662.2748 seconds
Running tokenization for UD_Greek-GUD
Total time taken: 688.8315 seconds


run pos for all languages

In [26]:
for UD_PATH in UD_PATHS:
  print(f"Running pos for {UD_PATH}")
  start_time = time.perf_counter()
  # changed log path
  !python3 -m stanza.utils.datasets.prepare_pos_treebank {UD_PATH}  &> logs/{UD_PATH}_prepare_pos_log.txt
  !python3 -m stanza.utils.training.run_pos {UD_PATH} --wordvec_pretrain_file fastText/embeds/{UD_PATH}.vec --no_pretrain --max_steps 500 &> logs/{UD_PATH}_run_pos_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running pos for UD_Italian-ISDT
Total time taken: 1222.1426 seconds
Running pos for UD_German-GSD
Total time taken: 1593.9633 seconds
Running pos for UD_Afrikaans-AfriBooms
Total time taken: 1209.3458 seconds
Running pos for UD_Greek-GUD
Total time taken: 339.3333 seconds


run depparser for all languages

In [27]:
for UD_PATH in UD_PATHS:
  print(f"Running depparser for {UD_PATH}")
  start_time = time.perf_counter()
  # changed the logs destination
  !python3 -m stanza.utils.datasets.prepare_depparse_treebank {UD_PATH}  &> logs/{UD_PATH}_prepare_depparse_log.txt
  !python3 -m stanza.utils.training.run_depparse {UD_PATH} --wordvec_pretrain_file fastText/embeds/{UD_PATH}.vec --no_pretrain --max_steps 500 &> logs/{UD_PATH}_run_depparse_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

Running depparser for UD_Italian-ISDT
Total time taken: 1753.0101 seconds
Running depparser for UD_German-GSD
Total time taken: 1230.3232 seconds
Running depparser for UD_Afrikaans-AfriBooms
Total time taken: 1562.6426 seconds
Running depparser for UD_Greek-GUD
Total time taken: 670.6214 seconds


### Parsing

In [28]:
import stanza
from stanza.utils.conll import CoNLL
import os
import shutil

/Users/maria/Documents/DPMS/dialectometry/dialectometry-distances/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


language code taken from [stanza repo](https://github.com/stanfordnlp/stanza/blob/516b07140fdfdafdebc813ed8f3253097da5efd3/stanza/models/common/constant.py#L105)

In [29]:
LANGUAGE_CODES = {
    "UD_Afrikaans": "af",
    "UD_German": "de",
    "UD_Italian": "it",
    "UD_Greek": "el",
}

In [30]:
def get_model_name(ud_path_name):
  language_code = LANGUAGE_CODES[ud_path_name.split("-")[0]]
  extra_arguments = ud_path_name.split("-")[1]
  return "_".join([language_code, extra_arguments.lower()])

In [31]:
# changed the path
source_dir = "stanza-train/data/udbase"
target_dir = "test_data"

os.makedirs(target_dir,exist_ok=True)
os.makedirs("parsed_data", exist_ok=True)

for UD_PATH in UD_PATHS:
  for filename in os.listdir(f"{source_dir}/{UD_PATH}"):
    if filename.endswith("test.conllu"):
      source_path = os.path.join(source_dir, UD_PATH, filename)
      target_path = os.path.join(target_dir, filename)
      shutil.copy2(source_path, target_path)
      print(f"Copied: {filename}")

Copied: it_isdt-ud-test.conllu
Copied: de_gsd-ud-test.conllu
Copied: af_afribooms-ud-test.conllu
Copied: el_gud-ud-test.conllu


In [32]:
def get_first_pt_file(directory):
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.pt')]
        if files:
            return os.path.join(directory, files[0])
    return None

In [35]:
# changed paths
for UD_PATH in UD_PATHS:
  model_name = get_model_name(UD_PATH)
  lang = model_name.split("_")[0]

  print(f"Working on model for {UD_PATH}")

  #f_charlm_dir = f"/root/stanza_resources/{lang}/forward_charlm"
  #b_charlm_dir = f"/root/stanza_resources/{lang}/backward_charlm"
  f_charlm_dir = f"/Users/maria/stanza_resources/{lang}/forward_charlm"
  b_charlm_dir = f"/Users/maria/stanza_resources/{lang}/backward_charlm"

  forward_charlm_path = get_first_pt_file(f_charlm_dir)
  backward_charlm_path = get_first_pt_file(b_charlm_dir)

  print(f"Found Forward CharLM: {forward_charlm_path}")
  print(f"Found Backward CharLM: {backward_charlm_path}")

  if os.path.exists(f'saved_models/tokenize/{model_name}_charlm_tokenizer.pt'):
    tokenize_model_path = f'saved_models/tokenize/{model_name}_charlm_tokenizer.pt'
  elif os.path.exists(f'saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'):
    tokenize_model_path = f'saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'
  else:
    raise FileNotFoundError("No tokenizer found")

  # changed path and name of the model
  if os.path.exists(f'saved_models/pos/{model_name}_charlm_tagger.pt'):
    pos_model_path = f'saved_models/pos/{model_name}_charlm_tagger.pt'
  elif os.path.exists(f'saved_models/pos/{model_name}_nocharlm_tagger.pt'):
     pos_model_path = f'saved_models/pos/{model_name}_nocharlm_tagger.pt'
  else:
    raise FileNotFoundError("No pos tagger found")

  if os.path.exists(f'saved_models/depparse/{model_name}_charlm_parser.pt'):
    depparse_model_path = f'saved_models/depparse/{model_name}_charlm_parser.pt'
  elif os.path.exists(f'saved_models/depparse/{model_name}_nocharlm_parser.pt'):
     depparse_model_path = f'saved_models/depparse/{model_name}_nocharlm_parser.pt'
  else:
    raise FileNotFoundError("No parser found")

  processors = "tokenize,pos,lemma,depparse"

  pipeline_config = {
      "lang": lang,
      "use_gpu": True,
      "processors": processors,
      "tokenize_model_path": tokenize_model_path,
      "pos_model_path": pos_model_path,
      "depparse_model_path": depparse_model_path,
      "tokenize_pretokenized": True,
      "lemma_use_identity": True,
  }

  if forward_charlm_path:
      pipeline_config["tokenize_forward_charlm_path"] = forward_charlm_path

  if backward_charlm_path:
      pipeline_config["tokenize_backward_charlm_path"] = backward_charlm_path


  nlp = stanza.Pipeline(**pipeline_config)

  for filename in os.listdir("test_data"):
      print(f'Working on file {filename}')
      file_path = os.path.join("test_data", filename)

      doc = CoNLL.conll2doc(file_path)
      doc = nlp(doc)
      CoNLL.write_doc2conll(doc, f"parsed_data/{model_name}-on-{filename.split('.')[0]}.conllu")

2026-05-16 14:02:45 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Working on model for UD_Italian-ISDT
Found Forward CharLM: /Users/maria/stanza_resources/it/forward_charlm/conll17.pt
Found Backward CharLM: /Users/maria/stanza_resources/it/backward_charlm/conll17.pt


2026-05-16 14:02:46 INFO: Downloaded file to /Users/maria/stanza_resources/resources.json
2026-05-16 14:03:02 INFO: Loading these models for language: it (Italian):
| Processor | Package                 |
---------------------------------------
| tokenize  | saved_mode...kenizer.pt |
| pos       | saved_mode..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | saved_mode..._parser.pt |

2026-05-16 14:03:02 WARNING: GPU requested, but is not available!
2026-05-16 14:03:02 INFO: Using device: cpu
2026-05-16 14:03:02 INFO: Loading: tokenize
2026-05-16 14:03:02 INFO: Loading: pos
2026-05-16 14:03:02 INFO: Loading: lemma
2026-05-16 14:03:02 INFO: Loading: depparse
2026-05-16 14:03:02 INFO: Done loading processors!


Working on file af_afribooms-ud-test.conllu
Working on file el_gud-ud-test.conllu
Working on file de_gsd-ud-test.conllu
Working on file it_isdt-ud-test.conllu


2026-05-16 14:03:17 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


...

Working on file af_afribooms-ud-test.conllu
Working on file el_gud-ud-test.conllu
Working on file de_gsd-ud-test.conllu
Working on file it_isdt-ud-test.conllu


2026-05-16 14:04:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Working on model for UD_Greek-GUD
Found Forward CharLM: None
Found Backward CharLM: None


2026-05-16 14:04:21 INFO: Downloaded file to /Users/maria/stanza_resources/resources.json
2026-05-16 14:04:31 INFO: Loading these models for language: el (Greek):
| Processor | Package                 |
---------------------------------------
| tokenize  | saved_mode...kenizer.pt |
| pos       | saved_mode..._tagger.pt |
| lemma     | gdt_nocharlm            |
| depparse  | saved_mode..._parser.pt |

2026-05-16 14:04:31 WARNING: GPU requested, but is not available!
2026-05-16 14:04:31 INFO: Using device: cpu
2026-05-16 14:04:31 INFO: Loading: tokenize
2026-05-16 14:04:31 INFO: Loading: pos
2026-05-16 14:04:31 INFO: Loading: lemma
2026-05-16 14:04:31 INFO: Loading: depparse
2026-05-16 14:04:31 INFO: Done loading processors!


Working on file af_afribooms-ud-test.conllu
Working on file el_gud-ud-test.conllu
Working on file de_gsd-ud-test.conllu
Working on file it_isdt-ud-test.conllu
